# Prédiction de l'inhibition de BSEP avec un SVM

**Machine Learning for Biology — classification à partir de fingerprints moléculaires**

Ce notebook suit le chemin complet : **données → SMILES → RDKit → fingerprints ECFP4 → SVM → évaluation**. Il est volontairement organisé en petites étapes pour relier chaque opération Python à une question biologique ou méthodologique.

![Vue d'ensemble](https://raw.githubusercontent.com/nounouci/MLB_Toxicity/main/assets/pipeline.svg)

### La question biologique

BSEP (*Bile Salt Export Pump*, codé par **ABCB11**) est un transporteur situé sur la membrane canaliculaire des hépatocytes. Il contribue à l'élimination des sels biliaires vers la bile. Certains médicaments peuvent inhiber ce transporteur, ce qui peut favoriser leur accumulation dans le foie.

On cherche ici à prédire **si une molécule inhibe BSEP**, et non à prédire toutes les formes possibles de toxicité hépatique.

La colonne `active` fournit la classe à apprendre :
- **0** : non-inhibiteur selon la définition du jeu de données ;
- **1** : inhibiteur selon la définition du jeu de données.

Le fichier public de Morgan–Warner comprend 660 composés. Les classes proviennent de mesures expérimentales publiées, avec des critères pouvant varier entre sources.


## 1. Préparer l'environnement

Ce notebook utilise cinq bibliothèques principales : `pandas` pour les tableaux, `RDKit` pour les molécules, `numpy` pour les vecteurs, `scikit-learn` pour le SVM et `matplotlib` pour les figures.

Dans Colab, on peut installer RDKit en exécutant la cellule suivante. En local, le fichier `requirements.txt` contient les dépendances.


In [ ]:
%pip -q install rdkit


In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay, precision_recall_curve
)
import joblib

GRAINE = 42
DOSSIER_FIGURES = Path("figures")
DOSSIER_RESULTATS = Path("outputs")
DOSSIER_FIGURES.mkdir(exist_ok=True)
DOSSIER_RESULTATS.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight"
})
print("Environnement prêt.")


## 2. Lire les données

On commence toujours par comprendre les colonnes avant d'entraîner un modèle.

Le fichier contient :
- `compound_name` : nom du composé ;
- `source` : origine de la mesure (Morgan ou Warner) ;
- `base_rdkit_smiles` : structure moléculaire sous forme de texte ;
- `relation` et `pIC50` : informations expérimentales, parfois censurées ;
- `active` : étiquette de classification.

**Attention :** la pIC50 sert à caractériser l'inhibition. La mettre dans le SVM créerait une fuite d'information, car elle est directement liée au label `active`.


In [ ]:
URL_CSV = (
    "https://raw.githubusercontent.com/ATOMScience-org/AMPL/master/"
    "atomsci/ddm/examples/BSEP/data/morgan_warner_combined_bsep_data.csv"
)
FICHIER_CSV = Path("data/morgan_warner_combined_bsep_data.csv")

if FICHIER_CSV.exists():
    df = pd.read_csv(FICHIER_CSV)
else:
    df = pd.read_csv(URL_CSV)

print("Dimensions du fichier :", df.shape)
print("\nCinq premières lignes :")
print(df.head().to_string(index=False))
print("\nValeurs manquantes :")
print(df.isna().sum())
print("\nRépartition des classes :")
print(df["active"].value_counts().sort_index())


### Première visualisation — Combien d'inhibiteurs ?

Avant toute modélisation, on regarde si les deux classes sont représentées de manière comparable. Une classe majoritaire peut rendre l'accuracy trompeuse.

On regarde aussi la provenance des composés : des protocoles expérimentaux différents peuvent introduire des différences entre les deux sources.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
n_classes = df["active"].value_counts().reindex([0, 1], fill_value=0)
axes[0].bar(["Non-inhibiteurs (0)", "Inhibiteurs (1)"], n_classes,
            color=["#6A9FBA", "#D6A161"], width=0.6)
axes[0].set_title("Répartition des classes")
axes[0].set_ylabel("Nombre de molécules")
for i, nombre in enumerate(n_classes):
    axes[0].text(i, nombre + 4, str(nombre), ha="center", fontweight="bold")

source_table = pd.crosstab(df["source"], df["active"])
source_table = source_table.reindex(columns=[0, 1], fill_value=0)
source_table.plot(kind="bar", stacked=True, ax=axes[1],
                  color=["#6A9FBA", "#D6A161"], rot=0)
axes[1].set_title("Classes par étude d'origine")
axes[1].set_xlabel("Source")
axes[1].set_ylabel("Nombre de molécules")
axes[1].legend(["Non-inhibiteur", "Inhibiteur"])
fig.tight_layout()
fig.savefig(DOSSIER_FIGURES / "01_classes_et_sources.png", dpi=180)
plt.show()
plt.close(fig)


## 3. Vérifier les SMILES avec RDKit

Un **SMILES** est une manière d'écrire la structure chimique d'une molécule avec des caractères.

Par exemple, `CCO` représente l'éthanol. RDKit transforme ce texte en objet moléculaire (atomes et liaisons). Le **SMILES canonique** est une écriture standardisée utile pour repérer les doublons.

On conserve uniquement les structures lisibles, puis on retire les éventuels doublons et les structures ayant des labels contradictoires. Cette vérification est importante avant de séparer train et test.


In [ ]:
donnees = df.dropna(subset=["base_rdkit_smiles", "active"]).copy()
donnees["active"] = pd.to_numeric(donnees["active"], errors="coerce")
donnees = donnees[donnees["active"].isin([0, 1])].copy()
donnees["active"] = donnees["active"].astype(int)

def smiles_canonique(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

donnees["canonical_smiles"] = donnees["base_rdkit_smiles"].apply(smiles_canonique)
n_invalides = donnees["canonical_smiles"].isna().sum()
donnees = donnees.dropna(subset=["canonical_smiles"]).copy()

n_labels = donnees.groupby("canonical_smiles")["active"].nunique()
ambigus = set(n_labels[n_labels > 1].index)
donnees = donnees[~donnees["canonical_smiles"].isin(ambigus)]
n_doublons = donnees.duplicated("canonical_smiles").sum()
donnees = donnees.drop_duplicates("canonical_smiles")
donnees = donnees.sort_values("canonical_smiles").reset_index(drop=True)

print("Molécules conservées :", len(donnees))
print("SMILES invalides :", n_invalides)
print("Structures avec labels contradictoires :", len(ambigus))
print("Doublons retirés :", n_doublons)
assert donnees["active"].nunique() == 2


### Voir des molécules, pas seulement des lignes de CSV

RDKit peut aussi dessiner les structures. Ces images permettent de vérifier visuellement que le jeu contient des molécules très différentes. La figure suivante montre six composés choisis dans le fichier : ce sont des **exemples de structures**, pas une preuve de leur activité.


In [ ]:
exemples = donnees.sample(6, random_state=GRAINE)
molecules = [Chem.MolFromSmiles(s) for s in exemples["canonical_smiles"]]
legendes = [
    f"{nom[:22]}  |  active={classe}"
    for nom, classe in zip(exemples["compound_name"], exemples["active"])
]
image_molecules = Draw.MolsToGridImage(
    molecules, molsPerRow=3, subImgSize=(290, 210), legends=legendes
)
# Selon le contexte, RDKit renvoie une image PIL ou un objet IPython.display.Image.
if hasattr(image_molecules, "save"):
    image_molecules.save(str(DOSSIER_FIGURES / "02_exemples_molecules.png"))
else:
    (DOSSIER_FIGURES / "02_exemples_molecules.png").write_bytes(image_molecules.data)
# Dans un notebook, cette dernière expression affiche la grille.
from IPython.display import display
display(image_molecules)

## 4. Quelques propriétés chimiques pour comprendre la biologie

Les fingerprints seront les **seules variables du SVM**, mais il est utile d'observer d'abord des propriétés facilement interprétables :

- **MolWt** : masse moléculaire ;
- **LogP** : estimation de la lipophilie ;
- **TPSA** : surface polaire topologique.

Ces propriétés peuvent influencer la façon dont les molécules interagissent avec les membranes ou les protéines. Attention : une différence entre classes n'établit pas à elle seule une relation causale.


In [ ]:
donnees["MolWt"] = [
    Descriptors.MolWt(Chem.MolFromSmiles(s))
    for s in donnees["canonical_smiles"]
]
donnees["LogP"] = [
    Descriptors.MolLogP(Chem.MolFromSmiles(s))
    for s in donnees["canonical_smiles"]
]
donnees["TPSA"] = [
    Descriptors.TPSA(Chem.MolFromSmiles(s))
    for s in donnees["canonical_smiles"]
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, colonne, titre in zip(
    axes,
    ["MolWt", "LogP", "TPSA"],
    ["Masse moléculaire", "Lipophilie (LogP)", "Surface polaire (TPSA)"]
):
    groupes = [
        donnees.loc[donnees["active"] == c, colonne].values
        for c in [0, 1]
    ]
    bp = ax.boxplot(groupes, patch_artist=True, showfliers=False)
    for boite, couleur in zip(bp["boxes"], ["#6A9FBA", "#D6A161"]):
        boite.set_facecolor(couleur)
    ax.set_xticks([1, 2], ["Non-inhib.", "Inhib."])
    ax.set_title(titre)
    ax.grid(axis="y", alpha=0.2)
fig.suptitle("Comparaison descriptive des propriétés moléculaires", fontsize=14)
fig.tight_layout()
fig.savefig(DOSSIER_FIGURES / "03_proprietes_chimiques.png", dpi=180)
plt.show()
plt.close(fig)


## 5. Vectorisation ECFP4 — du SMILES au vecteur de 1 024 bits

Le SVM ne peut pas utiliser directement du texte SMILES. Avec RDKit, on calcule un **fingerprint Morgan** de rayon 2, souvent appelé **ECFP4**.

Ce vecteur contient 1 024 cases, chacune valant 0 ou 1. Elles résument la présence d'environnements atomiques codés par le fingerprint (avec possibilité de collisions entre fragments).

Pour 660 molécules valides, `X` a donc 660 lignes × 1 024 colonnes. `y` contient les étiquettes 0 et 1.

Il n'est **pas nécessaire de découper les SMILES en tokens** pour cette méthode.


In [ ]:
generateur = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=1024
)

X = np.array([
    generateur.GetFingerprintAsNumPy(Chem.MolFromSmiles(s))
    for s in donnees["canonical_smiles"]
], dtype=np.uint8)
y = donnees["active"].to_numpy()

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("Vingt premiers bits :", X[0, :20])
assert X.shape == (len(donnees), 1024)

fig, ax = plt.subplots(figsize=(11, 4))
ax.imshow(X[:24, :90], aspect="auto", cmap="Blues", interpolation="nearest")
ax.set_title("Aperçu des fingerprints : 24 molécules, 90 premiers bits")
ax.set_xlabel("Position du bit")
ax.set_ylabel("Molécule")
fig.tight_layout()
fig.savefig(DOSSIER_FIGURES / "04_fingerprints.png", dpi=180)
plt.show()
plt.close(fig)


## 6. Séparer entraînement et test

On garde **80 % des molécules pour apprendre** et **20 % pour tester** le modèle une seule fois à la fin.

La séparation est **stratifiée** : elle conserve approximativement la proportion d'inhibiteurs dans les deux ensembles.

Pour comparer le SVM au Random Forest du groupe, il faut utiliser les **mêmes molécules de test**. Si un fichier `data/team_split.csv` est fourni, le code le réutilise. Sinon, il crée une partition reproductible avec la graine 42.


In [ ]:
fichier_partage = Path("data/team_split.csv")

if fichier_partage.exists():
    partage = pd.read_csv(fichier_partage)
    etiquettes = partage.set_index("canonical_smiles")["split"]
    attribution = donnees["canonical_smiles"].map(etiquettes)
    if attribution.isna().any() or not attribution.isin(["train", "test"]).all():
        raise ValueError("Le fichier team_split.csv doit attribuer train/test à chaque molécule.")
    train_idx = np.flatnonzero(attribution.to_numpy() == "train")
    test_idx = np.flatnonzero(attribution.to_numpy() == "test")
else:
    train_idx, test_idx = train_test_split(
        np.arange(len(donnees)), test_size=0.20,
        stratify=y, random_state=GRAINE
    )

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

splits = donnees[["canonical_smiles", "active"]].copy()
splits["split"] = "train"
splits.loc[test_idx, "split"] = "test"
splits.to_csv(DOSSIER_RESULTATS / "splits_bsep.csv", index=False)

print("Entraînement :", len(y_train), "molécules", np.bincount(y_train))
print("Test :", len(y_test), "molécules", np.bincount(y_test))
assert set(train_idx).isdisjoint(set(test_idx))


## 7. Qu'est-ce qu'un SVM ?

Un **Support Vector Machine** cherche une frontière séparant les deux classes. Les molécules proches de cette frontière influencent particulièrement sa position : ce sont les *vecteurs de support*.

Deux versions sont comparées :

- **SVM linéaire** : frontière linéaire dans l'espace des 1 024 bits ;
- **SVM RBF** : frontière non linéaire obtenue grâce à un noyau.

Les paramètres importants sont :
- `C` : compromis entre marge et erreurs d'entraînement ;
- `gamma` : portée de l'influence d'une molécule pour le noyau RBF ;
- `class_weight` : pondération éventuelle des deux classes.

On utilise aussi un **DummyClassifier**, qui ne regarde pas les structures : il sert de référence. Un vrai modèle devrait être nettement meilleur.

Les fingerprints étant déjà binaires, on ne standardise pas les colonnes comme on le ferait avec des descripteurs continus.


In [ ]:
validation = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=GRAINE
)

grilles = {
    "SVM linéaire": {
        "kernel": ["linear"],
        "C": [0.01, 0.1, 1, 10],
        "class_weight": [None, "balanced"]
    },
    "SVM RBF": {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.001, 0.01, 0.1],
        "class_weight": [None, "balanced"]
    }
}

recherches = {}
lignes_cv = []

for nom, grille in grilles.items():
    debut = time.perf_counter()
    recherche = GridSearchCV(
        SVC(cache_size=500), grille,
        cv=validation, scoring="roc_auc",
        n_jobs=-1, refit=True
    )
    recherche.fit(X_train, y_train)
    recherches[nom] = recherche
    i = recherche.best_index_
    lignes_cv.append({
        "model": nom,
        "cv_roc_auc_mean": recherche.cv_results_["mean_test_score"][i],
        "cv_roc_auc_std": recherche.cv_results_["std_test_score"][i],
        "best_params": json.dumps(recherche.best_params_, ensure_ascii=False),
        "search_seconds": round(time.perf_counter() - debut, 2)
    })
    print(nom, ": ROC-AUC CV =", round(recherche.best_score_, 3))
    print("Paramètres :", recherche.best_params_, "\n")

table_cv = pd.DataFrame(lignes_cv).sort_values("cv_roc_auc_mean", ascending=False)
table_cv.to_csv(DOSSIER_RESULTATS / "svm_benchmark_cv.csv", index=False)
print(table_cv[["model", "cv_roc_auc_mean", "cv_roc_auc_std", "best_params"]].to_string(index=False))


### Comparer les deux noyaux sans regarder le test

La courbe ou les barres suivantes utilisent **uniquement les scores de validation croisée**. C'est à partir de ces scores que l'on sélectionne le modèle. Les petites barres d'erreur représentent la variabilité entre les plis de validation.


In [ ]:
fig, ax = plt.subplots(figsize=(8.3, 4.2))
ax.barh(table_cv["model"], table_cv["cv_roc_auc_mean"],
        xerr=table_cv["cv_roc_auc_std"],
        color=["#257FA4", "#89B5C8"], capsize=5)
ax.set_xlim(0, 1)
ax.set_xlabel("ROC-AUC moyenne en validation croisée (± écart-type)")
ax.set_title("Benchmark interne : noyau linéaire ou RBF ?")
for i, ligne in enumerate(table_cv.itertuples()):
    ax.text(ligne.cv_roc_auc_mean + 0.015, i,
            f"{ligne.cv_roc_auc_mean:.3f}", va="center")
fig.tight_layout()
fig.savefig(DOSSIER_FIGURES / "05_comparaison_svm_cv.png", dpi=180)
plt.show()
plt.close(fig)

nom_retenu = table_cv.iloc[0]["model"]
modele = recherches[nom_retenu].best_estimator_
print("Modèle retenu :", nom_retenu)
print("Meilleurs paramètres :", recherches[nom_retenu].best_params_)


## 8. Évaluation finale sur les molécules de test

Maintenant seulement, on évalue le modèle choisi sur `X_test`.

- La **ROC-AUC** décrit la qualité du classement des inhibiteurs ;
- l'**Average Precision (AP)** résume la courbe précision–rappel ;
- le **rappel** mesure combien d'inhibiteurs sont retrouvés ;
- le **F1** combine précision et rappel.

Pour un outil de dépistage, les **faux négatifs** méritent une attention particulière : ce sont des inhibiteurs que le modèle n'a pas détectés.


In [ ]:
def calcul_scores(nom, vraies_classes, predictions, scores):
    tn, fp, fn, tp = confusion_matrix(
        vraies_classes, predictions, labels=[0, 1]
    ).ravel()
    return {
        "model": nom,
        "test_n": len(vraies_classes),
        "roc_auc": roc_auc_score(vraies_classes, scores),
        "pr_auc_ap": average_precision_score(vraies_classes, scores),
        "balanced_accuracy": balanced_accuracy_score(vraies_classes, predictions),
        "accuracy": accuracy_score(vraies_classes, predictions),
        "precision_1": precision_score(vraies_classes, predictions, zero_division=0),
        "recall_1": recall_score(vraies_classes, predictions, zero_division=0),
        "f1_1": f1_score(vraies_classes, predictions, zero_division=0),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }

dummy = DummyClassifier(strategy="prior")
dummy.fit(X_train, y_train)

pred_dummy = dummy.predict(X_test)
score_dummy = dummy.predict_proba(X_test)[:, 1]

pred_svm = modele.predict(X_test)
score_svm = modele.decision_function(X_test)

table_test = pd.DataFrame([
    calcul_scores("Dummy prior", y_test, pred_dummy, score_dummy),
    calcul_scores(nom_retenu, y_test, pred_svm, score_svm)
])
table_test.to_csv(DOSSIER_RESULTATS / "svm_benchmark_test.csv", index=False)
print(table_test.round(3).to_string(index=False))


### Courbes ROC et précision–rappel

Sur la courbe ROC, on souhaite une courbe proche du coin supérieur gauche. Sur la courbe précision–rappel, on souhaite garder une précision élevée tout en retrouvant les inhibiteurs.

**L'AP n'est pas exactement l'aire calculée par une intégration trapézoïdale de la courbe PR** : elle est calculée avec `average_precision_score`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(
    y_test, score_svm, name=nom_retenu, ax=axes[0], color="#177FA6"
)
RocCurveDisplay.from_predictions(
    y_test, score_dummy, name="Dummy", ax=axes[0], color="#B7A4A0"
)
axes[0].plot([0, 1], [0, 1], "--", color="gray", alpha=0.5)
axes[0].set_title("ROC — test final")

PrecisionRecallDisplay.from_predictions(
    y_test, score_svm, name=nom_retenu, ax=axes[1], color="#177FA6"
)
PrecisionRecallDisplay.from_predictions(
    y_test, score_dummy, name="Dummy", ax=axes[1], color="#B7A4A0"
)
axes[1].set_title("Précision–rappel — test final")
fig.tight_layout()
fig.savefig(DOSSIER_FIGURES / "06_roc_et_precision_rappel.png", dpi=180)
plt.show()
plt.close(fig)


### Matrice de confusion

On distingue quatre types de prédiction :
- **TN** : non-inhibiteur correctement identifié ;
- **FP** : fausse alerte ;
- **FN** : inhibiteur manqué ;
- **TP** : inhibiteur détecté.

Cette figure est indispensable pour interpréter un classificateur dans un contexte biologique.


In [ ]:
fig, ax = plt.subplots(figsize=(6.3, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_svm, labels=[0, 1],
    display_labels=["Non-inhibiteur", "Inhibiteur"],
    cmap="Blues", colorbar=False, ax=ax
)
ax.set_title("Matrice de confusion du SVM retenu")
fig.tight_layout()
fig.savefig(DOSSIER_FIGURES / "07_matrice_confusion.png", dpi=180)
plt.show()
plt.close(fig)


## 9. Peut-on retrouver davantage d'inhibiteurs ?

Le seuil de décision par défaut d'un SVM n'est pas forcément adapté à un **dépistage**. Si on abaisse le seuil, on peut détecter davantage d'inhibiteurs, mais on risque d'augmenter les fausses alertes.

Pour étudier ce compromis, on choisit un seuil en maximisant le **F2-score**, qui donne plus de poids au rappel qu'à la précision.

**Le seuil est choisi à partir des prédictions hors pli sur l'entraînement**, jamais à partir des labels du test. Cette analyse reste exploratoire car le choix d'hyperparamètres et celui du seuil utilisent le même jeu d'entraînement.


In [ ]:
scores_oof = cross_val_predict(
    modele, X_train, y_train,
    cv=validation, method="decision_function", n_jobs=-1
)
p, r, seuils = precision_recall_curve(y_train, scores_oof)

# Le dernier point de la courbe ne possède pas de seuil associé.
p, r = p[:-1], r[:-1]
f2 = np.divide(
    5 * p * r, 4 * p + r,
    out=np.zeros_like(p), where=(4 * p + r) > 0
)
seuil_f2 = float(seuils[np.argmax(f2)])
pred_f2 = (score_svm >= seuil_f2).astype(int)

scores_f2 = calcul_scores(
    nom_retenu + " (seuil F2)", y_test, pred_f2, score_svm
)
pd.DataFrame([scores_f2]).to_csv(
    DOSSIER_RESULTATS / "svm_test_seuil_f2.csv", index=False
)
print("Seuil F2 sélectionné sur le train :", round(seuil_f2, 4))
print(pd.DataFrame([scores_f2]).round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.4))
comparaison = [
    recall_score(y_test, pred_svm),
    recall_score(y_test, pred_f2)
]
ax.bar(["Seuil standard", "Seuil F2"], comparaison,
       color=["#6A9FBA", "#27A087"], width=0.55)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Rappel des inhibiteurs")
ax.set_title("Plus de détection, mais davantage de fausses alertes")
for i, valeur in enumerate(comparaison):
    ax.text(i, valeur + 0.025, f"{valeur:.1%}",
            ha="center", fontweight="bold")
fig.tight_layout()
fig.savefig(DOSSIER_FIGURES / "08_rappel_et_seuil.png", dpi=180)
plt.show()
plt.close(fig)
print("Faux positifs : standard =", int(((pred_svm == 1) & (y_test == 0)).sum()),
      "; seuil F2 =", int(((pred_f2 == 1) & (y_test == 0)).sum()))


## 10. Conserver les résultats et assurer la reproductibilité

Le benchmark doit pouvoir être reproduit. On enregistre :
- les **métriques** et meilleurs paramètres ;
- les **prédictions sur le test** ;
- les **SMILES canoniques du train et du test** ;
- le **modèle entraîné** (fichier local, non nécessaire dans GitHub).

Pour comparer avec un autre algorithme, le fichier `splits_bsep.csv` est plus important que le hasard d'un même `random_state` : il garantit que les molécules comparées sont identiques.


In [ ]:
predictions = donnees.iloc[test_idx][[
    "compound_name", "source", "canonical_smiles", "active"
]].copy()
predictions["pred_svm"] = pred_svm
predictions["score_svm"] = score_svm
predictions["pred_seuil_f2"] = pred_f2
predictions.to_csv(DOSSIER_RESULTATS / "svm_test_predictions.csv", index=False)

joblib.dump(
    {"modele": modele, "seuil_f2": seuil_f2,
     "fp_size": 1024, "radius": 2},
    DOSSIER_RESULTATS / "modele_svm_bsep.joblib"
)
print("Fichiers de résultats :", DOSSIER_RESULTATS.resolve())
print("Figures :", DOSSIER_FIGURES.resolve())


## 11. Discussion scientifique

**Ce qu'on peut interpréter :**

Un modèle qui obtient une ROC-AUC nettement supérieure à 0,5 arrive à distinguer les classes mieux qu'un classement aléatoire. Le rappel et la matrice de confusion précisent **quels types d'erreurs** il commet. Le noyau RBF peut mieux s'adapter à des relations non linéaires que le noyau linéaire.

**Ce qu'on ne peut pas conclure :**

1. La prédiction concerne **l'inhibition de BSEP**, pas toutes les formes de toxicité hépatique.
2. Le dataset réunit deux sources expérimentales qui n'ont pas exactement les mêmes seuils.
3. Le test est aléatoire : des structures chimiquement proches peuvent apparaître dans train et test.
4. Les performances sur une nouvelle famille chimique peuvent être moins bonnes.
5. Un score ne remplace jamais une mesure biologique.

**Pour prolonger l'étude :** comparer le même SVM avec une séparation par *scaffold* (squelette moléculaire), ou tester une seconde représentation moléculaire.

### Références
- McLoughlin et al. (2021), *Machine Learning Models to Predict Inhibition of the Bile Salt Export Pump*. DOI : https://doi.org/10.1021/acs.jcim.0c00950
- Dataset ATOMScience AMPL : https://github.com/ATOMScience-org/AMPL/tree/master/atomsci/ddm/examples/BSEP/data
- RDKit : https://www.rdkit.org/docs/
- scikit-learn SVC : https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html

---

**Bilan :** les fingerprints ECFP4 permettent de convertir des molécules en vecteurs numériques, et le SVM apprend à distinguer des inhibiteurs de BSEP à partir de ces vecteurs. Une évaluation rigoureuse exige de séparer entraînement et test, sélectionner les hyperparamètres par validation croisée et examiner les faux négatifs autant que les scores globaux.
